# LLM Evaluation with DeepEval

**Intro to DeepEval · Important metrics · RAG evaluation · Custom evals: G-Eval, DAG, Arena G-Eval**

---

### The idea in one line

> **DeepEval is unit testing, but for LLM output.**

You already trust tests for normal code:

```
   assert add(2, 2) == 4
```

You cannot write that for an LLM, because there are many correct answers.

So DeepEval replaces the exact match with a **judged score and a pass mark**:

```
   assert answer_relevancy(answer) >= 0.7
```

Same habit, same safety net, adapted to text that is never identical twice.

---

# Part 1 - Introduction to DeepEval and LLM Evaluation

### Why you cannot test an LLM the normal way

| Normal code                  | LLM output                                   |
| ---------------------------- | -------------------------------------------- |
| One correct answer           | Many correct answers, worded differently     |
| Same input gives same output | Same input gives a different output each run |
| Fails loudly with an error   | Fails quietly with a confident wrong answer  |
| `assert x == y` works        | `assert x == y` is useless                   |

### What DeepEval gives you

1. A **test case**: what went in, what came out, and what should have come out
2. A **metric**: something that scores that test case from 0 to 1
3. A **threshold**: the pass mark, so a score becomes a pass or a fail

```
                         +-----------------------------+
   LLMTestCase  ------>  |          Metric             |  ------> score 0.86
   input                 |  (an LLM judges the output) |          threshold 0.7
   actual_output         +-----------------------------+          PASSED
   expected_output
   retrieval_context
```

### DeepEval vs RAGAS

|                | RAGAS                            | DeepEval                                  |
| -------------- | -------------------------------- | ----------------------------------------- |
| Shape          | Score a **dataset**, get a table | Score **test cases**, get pass or fail    |
| Feels like     | A report card                    | A test suite                              |
| Custom metrics | Limited                          | **Its main strength**: G-Eval, DAG, Arena |
| Best at        | Measuring a RAG pipeline         | Testing any LLM behaviour, plus RAG       |

**Use both.** RAGAS to measure your RAG quality, DeepEval to assert that specific
behaviours hold before you ship.

### What it costs

The judge is an LLM, so **every metric on every test case is at least one API call**,
and asking for a `reason` adds another.

**DeepEval's default judge is an expensive model.** We pin `gpt-4o-mini` on every metric
in this notebook, which is roughly 15 times cheaper. Keep test sets small while learning.

---

### The test case: the one object everything uses

Every metric reads from an `LLMTestCase`. These are the fields that matter:

| Field               | What it holds                                 | Who fills it    |
| ------------------- | --------------------------------------------- | --------------- |
| `input`             | the question                                  | you             |
| `actual_output`     | what your app answered                        | your app        |
| `expected_output`   | the correct answer                            | you, in advance |
| `retrieval_context` | the documents your **retriever** found        | your app        |
| `context`           | the documents that **should** have been found | you             |

> **`retrieval_context` and `context` are not the same thing**, and mixing them up is the
> most common DeepEval mistake.
> `retrieval_context` is what your system really fetched.
> `context` is the ideal ground truth. Most RAG metrics want `retrieval_context`.

---

---

# Part 2 - Important DeepEval Metrics

### The two families

| Family             | Question it answers                     | Metrics                                                                 |
| ------------------ | --------------------------------------- | ----------------------------------------------------------------------- |
| **RAG metrics**    | Is my retrieval and answering any good? | Answer relevancy, Faithfulness, Contextual precision, Contextual recall |
| **Safety metrics** | Is this output harmful or wrong?        | Hallucination, Bias, Toxicity                                           |

### What each one needs

| Metric                      | Measures                                            | Needs                                         |
| --------------------------- | --------------------------------------------------- | --------------------------------------------- |
| `AnswerRelevancyMetric`     | Does the answer address the question?               | `input`, `actual_output`                      |
| `FaithfulnessMetric`        | Is the answer supported by the retrieved documents? | `input`, `actual_output`, `retrieval_context` |
| `ContextualPrecisionMetric` | Are the useful documents ranked near the top?       | plus `expected_output`                        |
| `ContextualRecallMetric`    | Did retrieval find everything needed?               | plus `expected_output`                        |
| `HallucinationMetric`       | Does the answer contradict the given context?       | `context`                                     |
| `BiasMetric`                | Is the answer biased?                               | `actual_output`                               |
| `ToxicityMetric`            | Is the answer rude or harmful?                      | `actual_output`                               |

All return **0 to 1**, but the direction is not the same for all of them:

| Metrics                                               | Direction           | Passing means        |
| ----------------------------------------------------- | ------------------- | -------------------- |
| The four RAG metrics                                  | higher is better    | `score >= threshold` |
| `HallucinationMetric`, `BiasMetric`, `ToxicityMetric` | **lower is better** | `score <= threshold` |

> **This catches people out.** For the safety metrics the threshold is a **maximum**,
> not a minimum. `ToxicityMetric(threshold=0.5)` means "fail if toxicity goes above 0.5".

### Threshold: turning a score into a decision

```python
AnswerRelevancyMetric(threshold=0.7, model=JUDGE)
```

Score at or above 0.7 passes, below fails. **That is what makes this a test suite
rather than a report.**

> **Always pass `model=`.** Left alone, DeepEval picks an expensive default judge.
> Every metric in this notebook pins the cheap one.

### Run one metric

---

RAG App with Deepeval

---

# Part 4 - Custom evals

The built-in metrics cover generic quality. They cannot know **your** rules:

* "never promise a refund without mentioning the 30 day limit"
* "always answer in under 50 words"
* "must include the ticket number"

For that you write a custom metric. DeepEval gives you three tools, and choosing
between them is the real skill.

| Tool             | You provide                                | The judge         |
| ---------------- | ------------------------------------------ | ----------------- |
| **G-Eval**       | a sentence describing what good looks like | decides freely    |
| **DAG**          | a decision tree of explicit checks         | follows your tree |
| **Arena G-Eval** | two or more outputs                        | picks a winner    |

### 4.1 - G-Eval: describe the rule in plain English

The easiest one. You write the criteria as a sentence, and an LLM scores against it.

`evaluation_params` tells it which fields to look at.

### 4.2 - DAG: a decision tree you control

G-Eval is one judgement call, so the score can wobble between runs.
A **DAG** breaks the decision into small yes or no steps that you define.

**The four building blocks:**

| Node                     | What it does                                                   |
| ------------------------ | -------------------------------------------------------------- |
| `BinaryJudgementNode`    | asks a yes or no question. Verdicts must be `True` and `False` |
| `NonBinaryJudgementNode` | asks a multiple choice question. Verdicts are strings          |
| `TaskNode`               | extracts or reshapes something first, before any judging       |

You build it top down with two methods:

| Method                                 | Use on           | Meaning                                    |
| -------------------------------------- | ---------------- | ------------------------------------------ |
| `.add_verdict(verdict=..., score=N)`   | a judgement node | this answer ends here with score N         |
| `.add_verdict(verdict=..., then=node)` | a judgement node | this answer continues to another question  |
| `.add_node(child)`                     | a `TaskNode`     | hand the extracted result to the next node |

The tree we are about to build:

```
                 Does it mention a number of days?
                          /              \
                       True             False
                        /                  \
        Does it say 7 business days?      score 0
              /            \
           True           False
            /                \
        score 10          score 2
```

> **Two things that trip people up.**
>
> 1. A `BinaryJudgementNode` needs `verdict=True` and `verdict=False`. The strings
>    `"yes"` and `"no"` raise
>    `ValueError: All children BinaryJudgementNode must have a boolean verdict.`
> 2. Older tutorials build nodes with `children=[VerdictNode(...)]`. That still works
>    but is **deprecated** and prints a warning. Use `.add_verdict()` as below.

> **Scores in a DAG are out of 10**, and DeepEval divides by 10 to give the usual
> 0 to 1 score. So `score=10` becomes 1.0 and `score=2` becomes 0.2.

> **Why bother with the tree?** Because the score is now **explainable and repeatable**.
> Anyone can read the tree and predict the score. With G-Eval you are trusting the judge's
> overall impression.

### 4.3 - Arena G-Eval: which output is better?

The other two ask *"is this good?"*. Arena asks *"which of these is better?"*

Use it to compare two prompts, two models, or two versions of your app on the same input.

**Two rules the contestants must follow:** every contestant needs a **unique name**, and
they must all share the **same `input`**. You are comparing answers to one question.

> Judging "A or B" is far more reliable than scoring each one alone, because two
> mediocre answers can both land on 0.75 and tell you nothing.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.environ.get("LLM_API_KEY"):
    print("API KEY loaded")

API KEY loaded


## Test Case

In [3]:
from deepeval.test_case import LLMTestCase

test_case = LLMTestCase(
    input = "How long do refund take",
    actual_output ="Refunds are processed in 7 business days",
    expected_output =  "Refund takes 7 business days.",
    retrieval_context = ["Refunds are processed within 7 businees days for orders under 30 days ago."]
)

print("input:", test_case.input)
print("output:", test_case.actual_output)
print("retrieval_context:", test_case.retrieval_context)

input: How long do refund take
output: Refunds are processed in 7 business days
retrieval_context: ['Refunds are processed within 7 businees days for orders under 30 days ago.']


## Metrices

In [4]:
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.models import LocalModel

JUDGE = os.getenv("LLM_MODEL")
LLM_BASE_URL = os.getenv("LLM_BASE_URL")
LLM_API_KEY = os.getenv("LLM_API_KEY")

judge_model = LocalModel(
    model=JUDGE,
    api_key=LLM_API_KEY,
    base_url=LLM_BASE_URL,
)

metric = AnswerRelevancyMetric(
    threshold = 0.7,
    model= judge_model,

)

metric.measure(test_case)
print("Score: ", metric.score)
print("Passed: ", metric.is_successful())
print("Reason: ", metric.reason)

/home/balaji/LLM/.venv/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter
support
  warnings.warn('install "ipywidgets" for Jupyter support')

Score:  1.0
Passed:  True
Reason:  The score is 1.00 because the answer directly addresses the question and contains no irrelevant statements.


In [5]:
from deepeval import evaluate
from deepeval.metrics import FaithfulnessMetric

good_case = LLMTestCase(
    input = "How long do refund take",
    actual_output ="Refunds are processed in 7 business days",
    retrieval_context = ["Refunds are processed within 7 businees days for orders under 30 days ago."]
)

bad_case = LLMTestCase(
    input = "How long do refund take",
    actual_output ="We refund instantly by UPI, and we also provide free vouchers.",
    retrieval_context = ["Refunds are processed within 7 businees days for orders under 30 days ago."]
)

results = evaluate(
    test_cases = [good_case, bad_case],
    metrics = [
        AnswerRelevancyMetric(
            threshold = 0.7,
            model= judge_model,
        ),

        FaithfulnessMetric(
            threshold = 0.7,
            model= judge_model,
        ),
    ]

)

✨ You're running DeepEval's latest Answer Relevancy Metric! (using openai/gpt-oss-20b (Local Model), strict=False,
async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using openai/gpt-oss-20b (Local Model), strict=False, 
async_mode=True)...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:            How long do refund take                                                                │
│  │     Actual Output:    We refund instantly by UPI, and we also provide free vouchers.                         │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric           ┃ Score ┃ Threshold ┃ Reason                                                    │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Answer Relevancy │ 0.50  │ 0.70      │ The score is 0.50 because the answer does not address     │
│              │                  │       │           │ refund time.                                              │
│        FAIL  │ Faithfulness     │ 0.50  │ 0.70      │ The score is 0.50 because the actual output claims        │
│              │                  │       │           │ refunds are instant via UPI, which contradicts the        │
│              │                  │       │           │ retrieval context stating refunds are processed within    │
│              │                  │       │           │ 7 business days.                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                     ┃ Average Score          ┃ Pass Rate                                    ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Answer Relevancy           │ 0.75                   │ 50.00% | passed=1 | failed=1                 │ 2         │
│  Faithfulness               │ 0.75                   │ 50.00% | passed=1 | failed=1                 │ 2         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=455329;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 18.07s | token cost: None)
» Test Results (2 total tests):
   » Pass Rate: 50.0% | Passed: 1 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

### RAG with DeepEval

In [6]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model_name=os.environ.get("LLM_MODEL"),
    api_key=os.environ.get("LLM_API_KEY"),
    base_url=os.environ.get("LLM_BASE_URL"),
    temperature=0.1, 
)

result = llm.invoke("reply with exactly: langsmith is listening.")
print(result.content)

langsmith is listening


In [7]:
from langchain_ollama import OllamaEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

documents = [
    """
    REFUND ELIGIBILITY

    Customers may request a refund within 30 days of the purchase date.
    To be eligible for a refund, the product must be unused and in its
    original condition. Digital products may be eligible for a refund
    only if they have not been substantially used or downloaded.
    """,

    """
    NON-REFUNDABLE ITEMS

    The following items are non-refundable:
    - Gift cards
    - Discounted or clearance items
    - Personalized products
    - Products damaged by the customer
    - Services that have already been fully completed
    """,

    """
    HOW TO REQUEST A REFUND

    To request a refund, contact our customer support team with your
    order number, registered email address, and reason for the refund.
    Refund requests are typically reviewed within 3 to 5 business days.
    """,

    """
    REFUND PROCESSING TIME

    Once a refund is approved, the refund will be processed to the
    original payment method. Credit and debit card refunds may take
    5 to 10 business days to appear. Bank transfer refunds may take
    up to 7 business days.
    """,

    """
    SUBSCRIPTION REFUND POLICY

    Customers may cancel their subscription at any time. Monthly
    subscription payments are generally non-refundable after the
    billing period has started. Annual subscriptions may be eligible
    for a partial refund if cancelled within 14 days of renewal.
    """,

    """
    DAMAGED OR INCORRECT PRODUCTS

    If you receive a damaged, defective, or incorrect product, contact
    customer support within 7 days of delivery. You may be eligible for
    a replacement or a full refund. Supporting photographs may be
    required to process the request.
    """,

    """
    LATE REFUND REQUESTS

    Refund requests submitted after the standard 30-day refund period
    are normally not accepted. Exceptions may be considered for
    technical errors, duplicate charges, or other special circumstances.
    """
]

embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")
vector_store = InMemoryVectorStore.from_texts(documents, embedding=embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

found = retriever.invoke("I need my money back, how long does it take?")

for doc in found:
    print("-", doc.page_content)

- 
    REFUND PROCESSING TIME

    Once a refund is approved, the refund will be processed to the
    original payment method. Credit and debit card refunds may take
    5 to 10 business days to appear. Bank transfer refunds may take
    up to 7 business days.
    
- 
    HOW TO REQUEST A REFUND

    To request a refund, contact our customer support team with your
    order number, registered email address, and reason for the refund.
    Refund requests are typically reviewed within 3 to 5 business days.
    


In [8]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([    
    ("system", "You are a support assisstant Answer only using the context below. \n\n{context}"),
    ("human", "{question}"),
])


In [9]:
questions = [
    "How long do refund takes?",
    "How fast is express shipping?",
    "When is support available?"
]

expected = [
    "Refunds are processed within business days",
    "Express shippings takes 1 businees days",
    "Support is available Monday to Friday, 9am to 6pm IST"
]

test_case = []

for question, expected_answer in zip(questions, expected):
    docs = retriever.invoke(question)
    contexts = []

    for doc in docs:
        contexts.append(doc.page_content)

    messages = prompt.format_messages(context=".".join(contexts), question=question)
    answer = llm.invoke(messages).content

    test_case.append(LLMTestCase(
        input = question,
        actual_output = answer,
        expected_output = expected_answer,
        retrieval_context = contexts
    ))
    print("Question: ", question)


Question:  How long do refund takes?
Question:  How fast is express shipping?
Question:  When is support available?


In [10]:
from deepeval.metrics import ContextualPrecisionMetric, ContextualRecallMetric

rag_metrics = [
        AnswerRelevancyMetric(
            threshold = 0.7,
            model= judge_model,
        ),

        FaithfulnessMetric(
            threshold = 0.7,
            model= judge_model,
        ),
    
        ContextualRecallMetric(
            threshold = 0.7,
            model= judge_model,
        ),

        ContextualPrecisionMetric(
            threshold = 0.7,
            model= judge_model,
        ),
    ]

result = evaluate(test_cases=test_case, metrics=rag_metrics)


✨ You're running DeepEval's latest Answer Relevancy Metric! (using openai/gpt-oss-20b (Local Model), strict=False,
async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using openai/gpt-oss-20b (Local Model), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using openai/gpt-oss-20b (Local Model), 
strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using openai/gpt-oss-20b (Local Model), 
strict=False, async_mode=True)...

/home/balaji/LLM/.venv/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter
support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:              How fast is express shipping?                                                        │
│  │     Actual Output:      I’m sorry, but I don’t have that information.                                        │
│  │     Expected Output:    Express shippings takes 1 businees days                                              │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        PASS  │ Answer Relevancy     │ 1.00  │ 0.70      │ The score is 1.00 because the answer directly a...    │
│        PASS  │ Faithfulness         │ 1.00  │ 0.70      │ The score is 1.00 because the actual output per...    │
│        FAIL  │ Contextual Recall    │ 0.00  │ 0.70      │ The score is 0.00 because the expected output         │
│              │                      │       │           │ sentence is not supported by any node in the          │
│              │                      │       │           │ retrieval context.                                    │
│        FAIL  │ Contextual Precision │ 0.00  │ 0.70      │ The score is 0.00 because the top-ranked node (rank   │
│              │                      │       │           │ 1) contains a reason that states "The first           │
│              │                      │       │           │ document discusses refund processing times, stating   │
│              │                      │       │           │ 'Credit and debit card refunds may take 5 to 10       │
│              │                      │       │           │ business days to appear' and 'Bank transfer refunds   │
│              │                      │       │           │ may take up to 7 business days,' which does not       │
│              │                      │       │           │ provide information about express shipping times."    │
│              │                      │       │           │ This indicates the node is irrelevant. The second     │
│              │                      │       │           │ node (rank 2) also contains a reason stating "The     │
│              │                      │       │           │ second document explains how to request a refund      │
│              │                      │       │           │ and notes that 'Refund requests are typically         │
│              │                      │       │           │ reviewed within 3 to 5 business days,' again          │
│              │                      │       │           │ unrelated to express shipping duration." Since both   │
│              │                      │       │           │ nodes are irrelevant and no relevant nodes appear     │
│              │                      │       │           │

⚠ WARNING: No hyperparameters logged.
» ]8;id=350125;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 29.75s | token cost: None)
» Test Results (3 total tests):
   » Pass Rate: 33.33% | Passed: 1 | Failed: 2

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

### Custom Eval

* G-Eval
* DAG
* Arena G-Eval

#### G-Eval

In [11]:
# Single criteria

from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams

politeness = GEval(
    name="politeness",
    criteria="decide if the polite and professional in tone",
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
    threshold=0.7,
    model=judge_model
)

rude_case = LLMTestCase(
    input="Where is my order?",
    actual_output="How should i know check it yourself."
)

politeness.measure(rude_case)
print("score: ", politeness.score)
print("passed: ", politeness.is_successful())
print("reason: ", politeness.reason)

score:  0.0
passed:  False
reason:  The actual output lacks any polite markers such as please or thank you, and it contains a dismissive tone that is unprofessional and rude. It does not match the neutral tone of the input and introduces unprofessional content, failing all evaluation steps.


In [12]:
# Multiple criteria

from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams

politeness = GEval(
    name="politeness",
    evaluation_steps=[
        "Check whether the answer mention refunds at all.",
        "If it does, check that it also states the 30 days limit.",
        "Penalise heavily if a refund is promised with no mention of the limit."
    ],
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
    threshold=0.7,
    model=judge_model
)

rude_case = LLMTestCase(
    input="Where is my order?",
    actual_output="How should i know check it yourself."
)

politeness.measure(rude_case)
print("score: ", politeness.score)
print("reason: ", politeness.reason)

score:  0.0
reason:  The response does not mention refunds at all, failing the first evaluation step. Consequently, it does not satisfy the requirement to state the 30‑day limit or any refund policy, resulting in a score of 0.


#### DAG

In [13]:
from deepeval.metrics import DAGMetric
from deepeval.metrics.dag import DeepAcyclicGraph, BinaryJudgementNode

says_seven_days = BinaryJudgementNode(
    criteria="Does the answer says 7 business days"
)
says_seven_days.add_verdict(verdict=True, score=10)
says_seven_days.add_verdict(verdict=False, score=2)


mention_7_days = BinaryJudgementNode(
    criteria="Does the answer mention a number of day at all"
)
mention_7_days.add_verdict(verdict=True, then=says_seven_days)
mention_7_days.add_verdict(verdict=False, score=2)

dag = DeepAcyclicGraph(root_nodes=[mention_7_days])
refund_dag = DAGMetric(
    name = "Refund answer check",
    dag = dag,
    threshold = 0.7,
    model = judge_model
)

In [17]:
answers = [
    "Refunds are processed within 7 business days",
    "Refunds usually take about 20 days",
    "We will sort out your refund soon."
]

for answer in answers:
    case = LLMTestCase(
        input="How long the refund take",
        actual_output=answer,
    )
    refund_dag.measure(case)
    print("Answer: ", answer),
    print("passsed: ", refund_dag.is_successful()),
    print("score: ", refund_dag.score)

Answer:  Refunds are processed within 7 business days
passsed:  False
score:  0.2


Answer:  Refunds usually take about 20 days
passsed:  False
score:  0.2


Answer:  We will sort out your refund soon.
passsed:  False
score:  0.2


#### Arena G-Evals

In [15]:
from deepeval.metrics import ArenaGEval
from deepeval.test_case import ArenaTestCase
from deepeval.test_case.arena_test_case import Contestant

question = "How long do refund take?"
arena_case = ArenaTestCase(contestants = [
    Contestant(
        name = "Short prompt",
        test_case = LLMTestCase(
            input = question,
            actual_output = "7 business days"
        )
    ),

    Contestant(
        name = "Detailed prompt",
        test_case = LLMTestCase(
            input = question,
            actual_output = "Refund are processed within 7 business days, for order placed under 30 days ago."
        )
    )
])


comparision = ArenaGEval(
    name = "Most helpful answer",
    criteria = "pick the answer that is clearer and gives the customer more usefull detail, without adding anything untrue",
    evaluation_params = [SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
    model = judge_model,
)

comparision.measure(arena_case)
print("Winner: ", comparision.winner)
print("Reason: ", comparision.reason)

Winner:  Detailed prompt
Reason:  Detailed prompt provides a concise yet richer detail by specifying that refunds are processed within 7 business days for orders placed under 30 days ago, offering actionable information that Short prompt lacks. Both responses are clear and factually accurate, but Detailed prompt’s additional context makes it the better choice per the evaluation steps.
